# 2. Index Taxonomies

Builds the **retrieval index** for every active schema in Lakebase
Postgres. The index powers both the `retrieval_llm` and `tsvector`
classification strategies in notebook 3.

What lands here:

* For each schema, a Postgres table `"<schema>"."<name>_text"` holding
  one row per leaf with a precomputed `tsvector` over its text
  (label, parent labels, definitions, synonyms when available).
* A GIN index on the `tsvector` column for fast `@@ to_tsquery(...)`
  lookups.
* SP grants so the deployed app can read the index.

This notebook does **NOT classify anything**. Notebook 3 calls
`src.categorize.classify_retrieval_llm` / `classify_tsvector` against
these tables.

Why index everything (not just UNSPSC)?
* Indexing is cheap (one table, one GIN index, one MERGE per schema).
* It gives every schema a `tsvector` baseline strategy for free.
* It enables future migration of medium schemas (e.g. `gl_map`) to
  `retrieval_llm` without further code changes — just flip
  `SchemaSpec.classify_strategy`.

Tested on Serverless v4.

In [ ]:
%pip install pyyaml openpyxl pydantic 'databricks-sdk>=0.50.0' psycopg2-binary
%restart_python

In [ ]:
import os, sys, uuid
sys.path.insert(0, os.path.abspath('..'))

from src.utils import get_spark
from src.config import load_config
from src.schemas import list_schemas
import pyspark.sql.functions as F

spark = get_spark()
config = load_config()

In [ ]:
dbutils.widgets.removeAll()
dbutils.widgets.text('catalog', config.catalog)
dbutils.widgets.text('schema', config.schema_name)
dbutils.widgets.text('lakebase_instance', config.lakebase_instance)

catalog = dbutils.widgets.get('catalog')
schema = dbutils.widgets.get('schema')
lakebase_instance = dbutils.widgets.get('lakebase_instance')
spark.sql(f'USE {catalog}.{schema}')

## Build search-text views in Delta

For each schema, materialize a `<name>_search` table with one row per
leaf and a `search_text` column concatenating every text signal in the
taxonomy (label, parent labels, definitions, synonyms). This is the
input to `to_tsvector` in Postgres.

In [ ]:
def _sanitize(name: str) -> str:
    import re
    return re.sub(r'[^a-zA-Z0-9_]+', '_', name).strip('_').lower()

for s in list_schemas():
    spec = s.spec
    raw_cols = [spec.label_column] + list(spec.level_columns) + list(spec.description_columns)
    # Use the *sanitized* column names because the taxonomy table was
    # written by notebook 0 with snake-cased columns.
    cols = [f"COALESCE(`{_sanitize(c)}`, '')" for c in raw_cols if c]
    text_expr = "CONCAT_WS(' | ', " + ", ".join(cols) + ")"
    spark.sql(f"""
    CREATE OR REPLACE TABLE {catalog}.{schema}.{s.name}_search AS
    SELECT code, label, level_path,
           {text_expr} AS search_text
    FROM {catalog}.{schema}.{s.name}_taxonomy
    """)
    n = spark.table(f'{catalog}.{schema}.{s.name}_search').count()
    print(f'  {s.name}_search: {n:,} rows')

## Connect to Lakebase

One psycopg2 connection helper, reused for the bulk insert + index
creation. The deployed app's service principal needs `SELECT` on each
`<name>_text` table — granted at the end of the per-schema loop.

In [ ]:
from databricks.sdk import WorkspaceClient
import psycopg2
from psycopg2.extras import execute_values

w = WorkspaceClient()

def pg_connect():
    inst = w.database.get_database_instance(name=lakebase_instance)
    cred = w.database.generate_database_credential(
        request_id=str(uuid.uuid4()), instance_names=[lakebase_instance]
    )
    me = w.current_user.me()
    user = me.emails[0].value if me.emails else me.user_name
    return psycopg2.connect(
        host=inst.read_write_dns, dbname=config.lakebase_dbname,
        user=user, password=cred.token, sslmode='require',
    )

SP_ROLE = 'aab57292-8e82-48bf-8e43-e8f43fe2743f'  # spend-app SP

## Materialize per-schema tsvector indexes

For each schema:
1. Create `"<schema>"."<name>_text"` with `(code PK, label, level_path, search_text, tsv)`.
2. Bulk-insert from the Delta `<name>_search` table.
3. `UPDATE ... SET tsv = to_tsvector('english', search_text)`.
4. `CREATE INDEX ... USING GIN (tsv)`.
5. Grant SELECT to the SP role.

Idempotent: re-running upserts on `code` (primary key) and refreshes
the `tsv` column where it's NULL.

In [ ]:
BATCH = 1000

for s in list_schemas():
    table = f'{s.name}_text'
    print(f'-- {table} --')
    with pg_connect() as conn, conn.cursor() as cur:
        cur.execute(f'CREATE SCHEMA IF NOT EXISTS "{schema}"')
        cur.execute(f'''
        CREATE TABLE IF NOT EXISTS "{schema}"."{table}" (
            code TEXT PRIMARY KEY,
            label TEXT,
            level_path TEXT[],
            search_text TEXT,
            tsv tsvector
        )''')
        cur.execute(
            f'CREATE INDEX IF NOT EXISTS "{table}_tsv_idx" '
            f'ON "{schema}"."{table}" USING GIN (tsv)'
        )
        conn.commit()

    iterator = (
        spark.table(f'{catalog}.{schema}.{s.name}_search')
             .select('code', 'label', 'level_path', 'search_text')
             .toLocalIterator()
    )
    n = 0
    upsert_sql = f'''
        INSERT INTO "{schema}"."{table}" (code, label, level_path, search_text) VALUES %s
        ON CONFLICT (code) DO UPDATE SET
            label = EXCLUDED.label,
            level_path = EXCLUDED.level_path,
            search_text = EXCLUDED.search_text,
            tsv = NULL
    '''
    with pg_connect() as conn, conn.cursor() as cur:
        batch = []
        for row in iterator:
            batch.append((
                row['code'], row['label'],
                list(row['level_path'] or []),
                row['search_text'],
            ))
            if len(batch) >= BATCH:
                execute_values(cur, upsert_sql, batch, template='(%s,%s,%s,%s)')
                conn.commit()
                n += len(batch)
                batch.clear()
        if batch:
            execute_values(cur, upsert_sql, batch, template='(%s,%s,%s,%s)')
            conn.commit()
            n += len(batch)
        cur.execute(
            f'UPDATE "{schema}"."{table}" SET tsv = to_tsvector(\'english\', search_text) '
            f'WHERE tsv IS NULL'
        )
        conn.commit()
        try:
            cur.execute(f'GRANT SELECT ON "{schema}"."{table}" TO "{SP_ROLE}"')
            conn.commit()
        except Exception as e:
            print(f'  grant skipped: {e}')
        cur.execute(f'SELECT COUNT(*) FROM "{schema}"."{table}"')
        rows = cur.fetchone()[0]
    print(f'  upserted={n:,} rows={rows:,}')